<a href="https://colab.research.google.com/github/MarceCorreal2/Robots-NT/blob/main/Copia_de_Procesamiento_Indicadores_Backtest_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Este cuaderno limpia y organiza los datos del resumen del strategy analyzer de los robots y debe incluir los indicadores en la
# tabla robots y procesar la calificación

## Procesamiento de Indicadores de Backtest

El objetivo de este cuaderno es tomar los datos crudos de los resúmenes del *strategy analyzer* de tus robots, limpiarlos y extraer los indicadores clave mencionados. Una vez procesados, los datos se guardarán en un formato limpio para futuros análisis.

In [ ]:
# Celda 1 - Setup, librerías, Autentificación

import pandas as pd
import os
import re
import datetime
import json
import gspread
from google.colab import auth
import google.auth

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Autenticar al usuario para acceder a Google Drive y Sheets
# Esto debería abrir una ventana del navegador para la autenticación si no se ha hecho recientemente
auth.authenticate_user()
creds, project = google.auth.default()
gc = gspread.authorize(creds)

# Nombre de la hoja de cálculo que el usuario mencionó, corregido con la acentuación
spreadsheet_name = "Evaluación Cuantitativa de Bots"

print("Libraries imported and pandas display options configured.")
print("Autenticación con Google Sheets completada.")

Libraries imported and pandas display options configured.


In [ ]:
# Celda 2- Libreria Mount Google Drive para Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [ ]:
# Celda 3 - Autenticar para Google Sheets y cargar la hoja de evaluación

import gspread
from google.colab import auth
import google.auth
import pandas as pd

# Autenticar al usuario para acceder a Google Drive y Sheets
# Esto debería abrir una ventana del navegador para la autenticación si no se ha hecho recientemente
auth.authenticate_user()
creds, project = google.auth.default()
gc = gspread.authorize(creds)

print("Autenticación con Google Sheets completada.")





Autenticación con Google Sheets completada.


In [ ]:
# Celda 4 — Conectar Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Celda 5 — Definir Variables

bot_name = 'RB001_MNQ_A'
test_number = 'T001' # <--- Modifica este valor para diferentes pruebas (ej. 'T002')
instrument = 'MNQ'

print(f"Bot Name: {bot_name}")
print(f"Test Number: {test_number}")
print(f"Instrument: {instrument}")

Bot Name: RB001_MNQ_A
Test Number: T001
Instrument: MNQ


In [ ]:
# Celda 6 — Definir Rutas dinámicas

RAW_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/'
CLEAN_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/'

print(f"Ruta de Datos Crudos: {RAW_DATA_PATH}")
print(f"Ruta de Datos Limpios: {CLEAN_DATA_PATH}")

Ruta de Datos Crudos: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/
Ruta de Datos Limpios: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/


In [ ]:
# Celda 7 - Cargar y mostrar un archivo CSV de ejemplo con parsing robusto

import os
import pandas as pd

# Construir el nombre del archivo dinámicamente usando las variables definidas en Celda 3
sample_file_name = f'SA_{bot_name}_{test_number}.csv'
sample_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

print(f"Cargando archivo de ejemplo: {sample_file_path}")

try:
    # Read the first few lines to understand the structure and find the actual data start
    raw_lines = []
    with open(sample_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to cover potential header length
            line = f.readline()
            if not line: # EOF
                break
            raw_lines.append(line.strip())

    print("\nPrimeras 20 líneas del archivo crudo para inspección:")
    for i, line in enumerate(raw_lines[:20]):
        print(f"Línea {i+1}: {line}")

    # Try to find the line that indicates the start of the actual performance metrics
    # Common indicators like 'Total net profit' usually appear at the start of data section.
    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row != -1:
        print(f"\nIdentificado el inicio de los datos de indicadores en la línea (0-index): {data_start_row}")
        # Read the CSV again, skipping lines up to the identified data start
        # We set header=None because the first column will contain the indicator names,
        # and the subsequent columns are values (e.g., All trades, Long trades, Short trades).
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # Assuming the first column is the indicator name and the next are its values
        # We need to clean up the column names based on the context.
        # Let's just display the raw parsed DataFrame for now.
        print("\nDataFrame de indicadores procesado (primeras 5 filas):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del DataFrame procesado:")
        print(sample_df.columns.tolist())
    else:
        print("\nNo se pudo identificar el inicio de los datos de indicadores ('Total net profit' no encontrado). Se muestra la lectura inicial sin procesar.")
        # Fallback if specific data start not found, try reading with just separator
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';')
        print("\nPrimeras 5 filas del archivo de ejemplo (lectura básica):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del archivo de ejemplo (lectura básica):")
        print(sample_df.columns.tolist())

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al leer el archivo CSV: {e}")

Cargando archivo de ejemplo: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/SA_RB001_MNQ_A_T001.csv

Primeras 20 líneas del archivo crudo para inspección:
Línea 1: Performance;All trades;Long trades;Short trades;
Línea 2: Total net profit;$ 213,00;$ 213,00;$ 0,00;
Línea 3: Gross profit;$ 2243,00;$ 2243,00;$ 0,00;
Línea 4: Gross loss;-$ 2030,00;-$ 2030,00;$ 0,00;
Línea 5: Commission;$ 95,00;$ 95,00;$ 0,00;
Línea 6: Profit factor;1,10;1,10;1,00;
Línea 7: Max drawdown;-$ 995,60;-$ 995,60;$ 0,00;
Línea 8: Sharpe ratio;2,29;2,29;1,00;
Línea 9: Sortino ratio;1,00;1,00;1,00;
Línea 10: Ulcer index;0,01;0,01;0,00;
Línea 11: R squared;0,18;0,18;0,00;
Línea 12: Total Fees;$ 0,00;$ 0,00;$ 0,00;
Línea 13: Probability;42,14 %;42,14 %;0,00 %;
Línea 14: ;;;;
Línea 15: Start date;25/05/2026;;;
Línea 16: Start time;12:00 AM;;;
Línea 17: End date;3/06/2026;;;
Línea 18: End time;12:00 AM;;;
Línea 19: ;;;;
Línea 20: Total # of trades;50;50;0;

Identificado el inicio de los datos

In [ ]:
#Celda 8 - Limpieza de datos


import os
import pandas as pd
import re
from datetime import datetime

# Lista para almacenar los DataFrames de indicadores de cada archivo
all_indicators_list = []

# Función para limpiar y convertir valores numéricos
def clean_numeric_value(value):
    if isinstance(value, str):
        value = value.replace('$', '').replace(' ', '').replace('%', '').replace(',', '.')
        if value == '' or value == '-':
            return None
        try:
            return float(value)
        except ValueError:
            return value
    return value

# Usar el sample_file_name y sample_file_path ya definidos en Celda 5
# para procesar solo el archivo deseado.
print(f"Procesando el archivo especificado: {sample_file_name}")

# full_file_path ya está definido en Celda 5 y es el que queremos procesar
# Construimos full_file_path nuevamente aquí para asegurar que sea el correcto
# si Celda 5 no se ejecuta justo antes.
full_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

try:
    # --- Parsing robusto (similar a Celda 5) ---
    raw_lines = []
    with open(full_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to find data start
            line = f.readline()
            if not line: break
            raw_lines.append(line.strip())

    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row == -1:
        print(f"Advertencia: No se encontró el inicio de datos para {sample_file_name}. No se procesará este archivo.")
    else:
        temp_df = pd.read_csv(full_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # --- Limpieza y extracción (similar a Celda 6)---
        # Aplicar nombres de columna iniciales y establecer índice
        temp_df.columns = ['Performance', 'All trades', 'Long trades', 'Short trades', 'Extra_Column']
        temp_df = temp_df.drop(columns=['Extra_Column'])
        temp_df['Performance'] = temp_df['Performance'].str.strip()
        temp_df = temp_df.set_index('Performance')

        # Aplicar limpieza a valores numéricos
        for col in ['All trades', 'Long trades', 'Short trades']:
            temp_df[col] = temp_df[col].apply(clean_numeric_value)

        def get_indicator_value(df, indicator_name):
            try:
                return df.loc[indicator_name.strip(), 'All trades']
            except KeyError:
                return None

        # Extraer los indicadores solicitados
        NetProfit = get_indicator_value(temp_df, 'Total net profit')
        PF = get_indicator_value(temp_df, 'Profit factor')
        WR_probability = get_indicator_value(temp_df, 'Probability')
        DD_max = get_indicator_value(temp_df, 'Max drawdown')
        # RecoveryFactor = get_indicator_value(temp_df, 'Recovery factor') # Original line, now modified
        TotalTrades = get_indicator_value(temp_df, 'Total # of trades')
        Winners = get_indicator_value(temp_df, 'Number of winning trades')
        GrossProfit = get_indicator_value(temp_df, 'Gross profit')
        GrossLoss = get_indicator_value(temp_df, 'Gross loss')
        AvgWinTrade = get_indicator_value(temp_df, 'Avg winning trade') # New indicator
        AvgLossTrade = get_indicator_value(temp_df, 'Avg losing trade') # New indicator

        WR = WR_probability if WR_probability is not None else \
             (Winners / TotalTrades) * 100 if TotalTrades and Winners is not None and TotalTrades != 0 else None

        # Corrected PayoffRatio calculation
        PayoffRatio = (AvgWinTrade / abs(AvgLossTrade)) if AvgWinTrade and AvgLossTrade and AvgLossTrade != 0 else None

        # Calculate RecoveryFactor as Net Profit / abs(Max Drawdown)
        RecoveryFactor = (NetProfit / abs(DD_max)) if NetProfit is not None and DD_max is not None and DD_max != 0 else None

        # Extracción de fechas y cálculo de Net Profit/Mes
        FechaInicio = None
        FechaFin = None
        for line in raw_lines:
            if 'Start date' in line:
                match = re.search(r'Start date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaInicio = datetime.strptime(match.group(1), '%d/%m/%Y')
            elif 'End date' in line:
                match = re.search(r'End date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaFin = datetime.strptime(match.group(1), '%d/%m/%Y')

        NumMonths = None
        NetProfitPerMonth = None
        if NetProfit is not None and FechaInicio is not None and FechaFin is not None:
            delta = FechaFin - FechaInicio
            if delta.days > 0:
                NumMonths = delta.days / 30.44
                if NumMonths > 0:
                    NetProfitPerMonth = NetProfit / NumMonths

        # Extraer Robot Base, Test ID e Instrumento del nombre del archivo
        # Ejemplo: SA-RB01-MNQ-A-T001.csv
        file_parts = sample_file_name.replace('.csv', '').split('_') # Changed to split by underscore
        robot_base = file_parts[1] if len(file_parts) > 1 else None
        test_id = file_parts[-1] if len(file_parts) > 0 else None # Assuming last part is Test ID
        instrument_from_file = file_parts[2] if len(file_parts) > 2 else None

        # Crear un diccionario con los indicadores para este archivo
        file_indicators = {
            'Archivo': sample_file_name,
            'Robot Base': robot_base,
            'Test ID': test_id,
            'Instrumento': instrument_from_file,
            'NetProfit': NetProfit,
            'PF': PF,
            'WR': WR,
            'DD max': DD_max,
            'Recovery Factor': RecoveryFactor,
            'PayoffRatio': PayoffRatio,
            '# Trades': TotalTrades,
            '# Meses': NumMonths,
            'Net Profit/Mes': NetProfitPerMonth,
            'Fecha-Inicio': FechaInicio.strftime('%Y-%m-%d') if FechaInicio else None,
            'Fecha-Fin': FechaFin.strftime('%Y-%m-%d') if FechaFin else None,
            'Avg Win': AvgWinTrade,
            'Avg Loss': AvgLossTrade
        }
        all_indicators_list.append(file_indicators)

        # --- Nuevo código para guardar el archivo limpio individual sin subdirectorios dinámicos ---
        individual_df = pd.DataFrame([file_indicators])
        base_file_name = os.path.splitext(sample_file_name)[0] # e.g., 'SA-RB01-MNQ-A-T001'
        cleaned_individual_file_name = f"{base_file_name}-Limpio.csv"

        # La carpeta de salida es directamente CLEAN_DATA_PATH
        dynamic_output_dir = CLEAN_DATA_PATH
        os.makedirs(dynamic_output_dir, exist_ok=True)

        individual_output_path = os.path.join(dynamic_output_dir, cleaned_individual_file_name)
        individual_df.to_csv(individual_output_path, index=False)
        print(f"Indicadores individuales guardados en: {individual_output_path}")
        # --- Fin del nuevo código ---

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al procesar el archivo {sample_file_name}: {e}")

# Convertir la lista de diccionarios a un DataFrame consolidado
if all_indicators_list:
    consolidated_df = pd.DataFrame(all_indicators_list)
    print("\n--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---")
    display(consolidated_df.head())

    # No guardar el DataFrame consolidado aquí ya que la Celda 9 se encarga de la tabla maestra.
else:
    print("No se pudieron procesar indicadores de ningún archivo.")

Procesando el archivo especificado: SA_RB001_MNQ_A_T001.csv
Indicadores individuales guardados en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/SA_RB001_MNQ_A_T001-Limpio.csv

--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---


,Archivo,Robot Base,Test ID,Instrumento,NetProfit,PF,WR,DD max,Recovery Factor,PayoffRatio,# Trades,# Meses,Net Profit/Mes,Fecha-Inicio,Fecha-Fin,Avg Win,Avg Loss
0,SA_RB001_MNQ_A_T001.csv,RB001,T001,MNQ,213.0,1.1,42.14,-995.6,0.213941,4.419704,50.0,0.295664,720.413333,2026-05-25,2026-06-03,224.3,-50.75


In [ ]:
# Celda 9 - Reordenamiento de datos finales

import pandas as pd
import os

# Check if consolidated_df exists. If not, try to reconstruct it from the last saved individual cleaned file.
if 'consolidated_df' not in locals() and 'consolidated_df' not in globals():
    print("Advertencia: 'consolidated_df' no definido en el estado actual del kernel. Intentando cargar el último archivo limpio individual.")
    try:
        # Assuming bot_name, test_number, and CLEAN_DATA_PATH are defined in previous cells and are accessible.
        cleaned_individual_file_name = f"SA_{bot_name}_{test_number}-Limpio.csv"
        individual_output_path = os.path.join(CLEAN_DATA_PATH, cleaned_individual_file_name)

        if os.path.exists(individual_output_path):
            consolidated_df = pd.read_csv(individual_output_path)
            print(f"Éxito: 'consolidated_df' cargado desde {individual_output_path}.")
        else:
            print(f"Error: No se pudo cargar 'consolidated_df'. El archivo '{individual_output_path}' no existe. Por favor, asegúrese de ejecutar la 'Celda 6' primero.")
            consolidated_df = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
    except NameError as e:
        print(f"Error de variable al intentar cargar 'consolidated_df': {e}. Asegúrese de que 'bot_name', 'test_number' y 'CLEAN_DATA_PATH' estén definidos en celdas anteriores.")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort
    except Exception as e:
        print(f"Error inesperado al cargar 'consolidated_df': {e}")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort


# Proceed only if consolidated_df is not empty after the potential loading attempt
if not consolidated_df.empty:
    # Renombrar columnas para que coincidan exactamente con la solicitud del usuario
    # Make a copy to avoid SettingWithCopyWarning later, especially during column renames
    consolidated_df = consolidated_df.copy().rename(columns={
        'Fecha-Inicio': 'Fecha Inicio',
        'Fecha-Fin': 'Fecha Fin',
        'DD max': 'DD Max',
        'Net Profit/Mes': 'Net Profit/Mes'
    })

    # Definir el orden de las columnas solicitado por el usuario, incluyendo 'Robot Base' y 'Test ID'
    column_order = [
        'Fecha Inicio',
        'Fecha Fin',
        'Instrumento',
        'Robot Base', # Agregado para identificación única
        'Test ID',    # Agregado para identificación única
        '# Meses',
        '# Trades',
        'NetProfit',
        'Net Profit/Mes',
        'PF',
        'WR',
        'DD Max',
        'Recovery Factor',
        'PayoffRatio',
        'Avg Win',
        'Avg Loss'
    ]

    # Seleccionar y reordenar las columnas del DataFrame
    # Filter column_order to only include columns actually present in consolidated_df
    actual_columns_in_order = [col for col in column_order if col in consolidated_df.columns]
    final_df = consolidated_df[actual_columns_in_order]

    print("\n--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---")
    display(final_df.head())
else:
    final_df = pd.DataFrame() # Ensure final_df is defined even if consolidated_df is empty
    print("No se pudo generar 'final_df' porque 'consolidated_df' está vacío o no se pudo cargar.")


--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---


,Fecha Inicio,Fecha Fin,Instrumento,Robot Base,Test ID,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,2026-05-25,2026-06-03,MNQ,RB001,T001,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,0.213941,4.419704,224.3,-50.75


In [ ]:
# Celda 10 - Verificar tipos de datos del DataFrame final
print("\n--- Tipos de datos del DataFrame final: ---")
display(final_df.dtypes)


--- Tipos de datos del DataFrame final: ---


,0
Fecha Inicio,object
Fecha Fin,object
Instrumento,object
Robot Base,object
Test ID,object
# Meses,float64
# Trades,float64
NetProfit,float64
Net Profit/Mes,float64
PF,float64


In [ ]:
# Celda 11  - Ingesta datos en Tabla Maestra

MASTER_TABLE_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv'

# Helper function to extract Robot and Test ID from a string, designed to handle various formats
def parse_robot_test_id(s):
    if pd.isna(s):
        return None, None
    s = str(s).strip()

    robot_extracted = None
    test_id_extracted = None

    # Try to extract Robot (e.g., RB001)
    match_robot = re.search(r'RB(\d+)', s, re.IGNORECASE)
    if match_robot:
        robot_extracted = f"RB{match_robot.group(1).zfill(3)}" # Ensure 3 digits, e.g., RB001

    # Try to extract Test ID (e.g., T001)
    match_test_id = re.search(r'T(\d+)', s, re.IGNORECASE)
    if match_test_id:
        test_id_extracted = f"T{match_test_id.group(1).zfill(3)}" # Ensure 3 digits, e.g., T001

    return robot_extracted, test_id_extracted


# Renombrar columnas en final_df antes de cualquier otra operación para asegurar consistencia
# Usamos .copy() para evitar SettingWithCopyWarning
final_df_to_add = final_df.rename(columns={'Robot Base': 'Robot', 'Test ID': 'Numero del Test'}).copy()

# Asegurar que las columnas clave sean de tipo string para la comparación
final_df_to_add['Robot'] = final_df_to_add['Robot'].astype(str)
final_df_to_add['Numero del Test'] = final_df_to_add['Numero del Test'].astype(str)

# Convertir las columnas de fecha en final_df_to_add a tipo datetime para compatibilidad
for col in ['Fecha Inicio', 'Fecha Fin']:
    if col in final_df_to_add.columns:
        final_df_to_add.loc[:, col] = pd.to_datetime(final_df_to_add[col], errors='coerce')


# Verificar si el archivo de la tabla maestra existe
if os.path.exists(MASTER_TABLE_PATH):
    print(f"Cargando tabla maestra desde: {MASTER_TABLE_PATH}")
    master_df = pd.read_csv(MASTER_TABLE_PATH)

    # --- START: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---
    # First, ensure 'Robot' and 'Numero del Test' columns exist, possibly from old names
    if 'Robot Base' in master_df.columns and 'Robot' not in master_df.columns:
        master_df = master_df.rename(columns={'Robot Base': 'Robot'})
    if 'Test ID' in master_df.columns and 'Numero del Test' not in master_df.columns:
        master_df = master_df.rename(columns={'Test ID': 'Numero del Test'})

    # If the columns still don't exist, create them with placeholder
    if 'Robot' not in master_df.columns:
        master_df['Robot'] = pd.NA
    if 'Numero del Test' not in master_df.columns:
        master_df['Numero del Test'] = pd.NA

    # Apply robust parsing to standardize Robot and Numero del Test
    standardized_robot_col = []
    standardized_test_id_col = []

    for idx, row in master_df.iterrows():
        # Get current values, try to use existing if they seem valid
        current_robot_val = row['Robot']
        current_test_id_val = row['Numero del Test']

        # Attempt to parse from current 'Robot' value
        parsed_r_from_robot, parsed_t_from_robot = parse_robot_test_id(current_robot_val)
        # Attempt to parse from current 'Numero del Test' value
        parsed_r_from_test_id, parsed_t_from_test_id = parse_robot_test_id(current_test_id_val)
        # Attempt to parse from 'Archivo' column if it exists and current keys are problematic
        parsed_r_from_file = None
        parsed_t_from_file = None
        if 'Archivo' in master_df.columns and (pd.isna(current_robot_val) or pd.isna(current_test_id_val) or not str(current_robot_val).startswith('RB') or not str(current_test_id_val).startswith('T')):
             parsed_r_from_file, parsed_t_from_file = parse_robot_test_id(row['Archivo'])


        # Prioritize values that look correct (e.g., start with 'RB'/'T')
        # Combine parsed results, prioritizing from specific columns or more complete sources
        final_robot = None
        if parsed_r_from_robot and parsed_r_from_robot.startswith('RB'):
            final_robot = parsed_r_from_robot
        elif parsed_r_from_test_id and parsed_r_from_test_id.startswith('RB'):
            final_robot = parsed_r_from_test_id
        elif parsed_r_from_file and parsed_r_from_file.startswith('RB'):
            final_robot = parsed_r_from_file

        final_test_id = None
        if parsed_t_from_test_id and parsed_t_from_test_id.startswith('T'):
            final_test_id = parsed_t_from_test_id
        elif parsed_t_from_robot and parsed_t_from_robot.startswith('T'):
            final_test_id = parsed_t_from_robot
        elif parsed_t_from_file and parsed_t_from_file.startswith('T'):
            final_test_id = parsed_t_from_file

        standardized_robot_col.append(final_robot if final_robot else 'UNKNOWN_ROBOT')
        standardized_test_id_col.append(final_test_id if final_test_id else 'UNKNOWN_TEST')

    master_df['Robot'] = standardized_robot_col
    master_df['Numero del Test'] = standardized_test_id_col

    # Ensure 'Robot' and 'Numero del Test' are string type after standardization
    master_df['Robot'] = master_df['Robot'].astype(str)
    master_df['Numero del Test'] = master_df['Numero del Test'].astype(str)
    # --- END: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---

    # --- START: Filter out non-identifiable rows from the loaded master_df ---
    initial_rows = len(master_df)
    master_df = master_df[
        (master_df['Robot'] != 'UNKNOWN_ROBOT') &
        (master_df['Numero del Test'] != 'UNKNOWN_TEST')
    ].copy() # Use .copy() to avoid SettingWithCopyWarning
    if len(master_df) < initial_rows:
        print(f"Advertencia: Se eliminaron {initial_rows - len(master_df)} filas no identificables (UNKNOWN_ROBOT/UNKNOWN_TEST) de la tabla maestra cargada.")
    # --- END: Filter out non-identifiable rows ---

    # --- ADDED: Clean up potentially erroneous columns from loaded master_df BEFORE merging ---
    # Drop 'NetProfit/Mes' (without space) if 'Net Profit/Mes' (with space) exists, as the latter is correct.
    if 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' in master_df.columns:
        print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) de la tabla maestra cargada.")
        master_df = master_df.drop(columns=['NetProfit/Mes'])
    # Rename 'NetProfit/Mes' (without space) to 'Net Profit/Mes' (with space) if only the former exists.
    elif 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' not in master_df.columns:
        print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en la tabla maestra cargada.")
        master_df = master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})

    # Drop 'Calificación Final' if it exists in the loaded master_df, as it will be recalculated later.
    if 'Calificación Final' in master_df.columns:
        print("Eliminando columna 'Calificación Final' de la tabla maestra cargada para recalcularla.")
        master_df = master_df.drop(columns=['Calificación Final'])
    # --- END ADDED CLEANUP ---

    # Convertir las columnas de fecha en master_df a tipo datetime si es necesario
    for col in ['Fecha Inicio', 'Fecha Fin']:
        if col in master_df.columns:
            # Convert to string first to avoid errors with mixed types, then to datetime
            master_df[col] = master_df[col].astype(str)
            master_df.loc[:, col] = pd.to_datetime(master_df[col], errors='coerce')


    print("\n--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---")
    display(master_df.head())

    # Get all unique (Robot, Numero del Test) pairs from the current processed data
    processed_keys = final_df_to_add[['Robot', 'Numero del Test']].drop_duplicates()

    # Create a boolean mask to identify rows in master_df that should be removed
    # These are rows whose (Robot, Numero del Test) pair is present in processed_keys
    mask_to_remove = master_df.set_index(['Robot', 'Numero del Test']).index.isin(
        processed_keys.set_index(['Robot', 'Numero del Test']).index
    )

    # Filter master_df to remove rows that are present in the current processed data
    master_df_filtered = master_df[~mask_to_remove].copy()

    # Concatenate the filtered master_df with the new records
    updated_master_df = pd.concat([master_df_filtered, final_df_to_add], ignore_index=True)

else:
    print(f"Advertencia: La tabla maestra no existe en {MASTER_TABLE_PATH}. Creando una nueva tabla maestra con los datos actuales.")
    updated_master_df = final_df_to_add.copy() # La primera entrada será la ejecución actual

# --- NUEVA LÓGICA DE LIMPIEZA DE COLUMNAS SIMILARES (retained for final check) ---
# This block acts as a fallback to ensure that after concatenation,
# if 'NetProfit/Mes' (without space) is still present and 'Net Profit/Mes' (with space) also exists,
# the former is removed. This might catch issues if final_df_to_add somehow introduced it, though unlikely.
if 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' in updated_master_df.columns:
    print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) duplicada en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.drop(columns=['NetProfit/Mes'])
elif 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' not in updated_master_df.columns:
    print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})
# --- FIN NUEVA LÓGICA DE LIMPIEZA ---


# Reordenar columnas para colocar 'Robot' y 'Numero del Test' al principio
expected_cols_order = [
    'Robot', 'Numero del Test', 'Fecha Inicio', 'Fecha Fin', 'Instrumento',
    '# Meses', '# Trades', 'NetProfit', 'Net Profit/Mes', 'PF', 'WR', # Asegurar que el nombre aquí sea el correcto
    'DD Max', 'Recovery Factor', 'PayoffRatio', 'Avg Win', 'Avg Loss'
]
# Add any missing columns to updated_master_df that are in expected_cols_order, filling with NaN
for col in expected_cols_order:
    if col not in updated_master_df.columns:
        updated_master_df[col] = pd.NA

# Reorder columns based on expected_cols_order
updated_master_df = updated_master_df[expected_cols_order]


print("\n--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---")
display(updated_master_df.head())

# Guardar la tabla maestra actualizada
updated_master_df.to_csv(MASTER_TABLE_PATH, index=False)
print(f"\nTabla maestra actualizada y guardada en: {MASTER_TABLE_PATH}")

Cargando tabla maestra desde: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv

--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,RB001,T001,2026-05-25 00:00:00,2026-06-03 00:00:00,MNQ,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,0.213941,4.419704,224.3,-50.75



--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,RB001,T001,2026-05-25 00:00:00,2026-06-03 00:00:00,MNQ,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,0.213941,4.419704,224.3,-50.75



Tabla maestra actualizada y guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv


In [ ]:
# Celda 11 - Verificar datos

MASTER_TABLE_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv'

print(f"Verificando el contenido de: {MASTER_TABLE_PATH}")
if os.path.exists(MASTER_TABLE_PATH):
    verified_master_df = pd.read_csv(MASTER_TABLE_PATH)
    print("\n--- Contenido del archivo de la Tabla Maestra (primeras 5 filas): ---")
    display(verified_master_df.head())
    print(f"Total de filas en la tabla maestra: {len(verified_master_df)}")
else:
    print("El archivo de la Tabla Maestra no se encontró.")

Verificando el contenido de: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv

--- Contenido del archivo de la Tabla Maestra (primeras 5 filas): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,RB001,T001,2026-05-25 00:00:00,2026-06-03 00:00:00,MNQ,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,0.213941,4.419704,224.3,-50.75


Total de filas en la tabla maestra: 1


In [ ]:
# Celda 12 - Cargar y mostrar la hoja de evaluación cuantitativa (corregido el parseo de cabecera)

# La autenticación (gc) ya está definida en la Celda 1.
# Definimos el nombre de la hoja de cálculo aquí para asegurar que sea el correcto.
spreadsheet_name = "Evaluación Cuantitativa de Bots"

try:
    spreadsheet = gc.open(spreadsheet_name)
    worksheet = spreadsheet.sheet1 # Asumir que la primera hoja es la relevante

    # Obtener todos los valores de la hoja
    data = worksheet.get_all_values()

    if data:
        # Buscar la fila que contiene 'Indicador' para usarla como encabezado
        header_row_index = -1
        for i, row in enumerate(data):
            if 'Indicador' in row:
                header_row_index = i
                break

        if header_row_index != -1:
            headers = data[header_row_index] # La fila identificada es el encabezado
            # Los datos comienzan después de la fila del encabezado
            evaluation_criteria_df = pd.DataFrame(data[header_row_index + 1:], columns=headers)

            # Limpiar los nombres de las columnas para eliminar espacios extra si los hay
            evaluation_criteria_df.columns = evaluation_criteria_df.columns.str.strip()

            print(f"\nHoja de cálculo '{spreadsheet_name}' abierta exitosamente.")
            print("\nPrimeras 5 filas de la hoja de evaluación como DataFrame para análisis:")
            display(evaluation_criteria_df.head())
            print(f"\nColumnas detectadas en la hoja de evaluación: {evaluation_criteria_df.columns.tolist()}")
        else:
            print("Error: No se encontró la fila con 'Indicador' en la hoja de cálculo. No se pudieron extraer los encabezados.")
            evaluation_criteria_df = pd.DataFrame() # Crear un DataFrame vacío para evitar errores posteriores
    else:
        print(f"La hoja de cálculo '{spreadsheet_name}' está vacía o no tiene datos.")
        evaluation_criteria_df = pd.DataFrame() # Crear un DataFrame vacío

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: La hoja de cálculo '{spreadsheet_name}' no se encontró. Por favor, verifica el nombre y que tengas permisos de acceso.")
    evaluation_criteria_df = pd.DataFrame() # Crear un DataFrame vacío
except Exception as e:
    print(f"Ocurrió un error al leer la hoja de cálculo: {e}")
    evaluation_criteria_df = pd.DataFrame() # Crear un DataFrame vacío


Hoja de cálculo 'Evaluación Cuantitativa de Bots' abierta exitosamente.

Primeras 5 filas de la hoja de evaluación como DataFrame para análisis:


,Indicador,Porcentaje de ponderación,0,1,2,3
0,Net Profit/Mes,30,<100,200-500,500-1000,>1000
1,Profit Factor (PF),25,< 1.20,1.20 – 1.49,1.50 – 1.99,≥ 2.00
2,Winning Rate (WR),15,< 35%,35% – 49%,50% – 64%,≥ 65%
3,Recovery Factor (RF),15,< 1.0,1.0 – 1.99,2.0 – 3.99,≥ 4.0
4,Payoff Ratio,15,< 1.20,1.20 – 1.79,1.80 – 2.99,≥ 3.00



Columnas detectadas en la hoja de evaluación: ['Indicador', 'Porcentaje de ponderación', '0', '1', '2', '3']


## Aplicación de Criterios de Evaluación y Calificación de Bots

Este paso se enfoca en usar la `evaluation_criteria_df` para calificar los indicadores de cada bot en la `master_df` y calcular una puntuación final ponderada.

In [ ]:
# Celda 13 - Preparar los criterios de evaluación y definir la función de scoring

import re

# Crear un diccionario para almacenar los criterios de evaluación en un formato más manejable
# Key: Nombre del indicador (ajustado para coincidir con master_df)
# Value: {'weight': ponderacion, 'ranges': [(min, max, score)]}

evaluation_rules = {}

# Mapeo manual de nombres de indicadores de la hoja de evaluación a la tabla maestra
indicator_name_map = {
    'Net Profit/Mes': 'Net Profit/Mes',
    'Profit Factor (PF)': 'PF',
    'Winning Rate (WR)': 'WR',
    'Recovery Factor (RF)': 'Recovery Factor',
    'Payoff Ratio': 'PayoffRatio'
}

# Función para parsear las cadenas de criterios en rangos numéricos
def parse_range_string(range_str):
    range_str = range_str.replace(' ', '').replace('–', '-') # Normalizar espacios y guiones
    range_str = range_str.replace('%', '') # Eliminar el signo de porcentaje

    if '<' in range_str:
        value = float(range_str.replace('<', ''))
        return -float('inf'), value # Menos infinito hasta el valor
    elif '>' in range_str:
        value = float(range_str.replace('>', ''))
        return value, float('inf') # Desde el valor hasta infinito
    elif '≥' in range_str:
        value = float(range_str.replace('≥', ''))
        return value, float('inf') # Desde el valor (incluido) hasta infinito
    elif '-' in range_str:
        parts = range_str.split('-')
        if len(parts) == 2:
            min_val = float(parts[0])
            max_val = float(parts[1])
            return min_val, max_val
    return None, None # En caso de no match

for index, row in evaluation_criteria_df.iterrows():
    indicator_eval_name = row['Indicador'].strip()
    if indicator_eval_name in indicator_name_map:
        master_df_col_name = indicator_name_map[indicator_eval_name]
        weight_str = str(row['Porcentaje de ponderación']).replace('%', '').strip()
        weight = int(weight_str) if weight_str.isdigit() else 0

        ranges_for_indicator = []
        for score_col in ['0', '1', '2', '3']:
            range_str = str(row[score_col]).strip()
            min_val, max_val = parse_range_string(range_str)
            if min_val is not None and max_val is not None:
                ranges_for_indicator.append((min_val, max_val, int(score_col)))

        evaluation_rules[master_df_col_name] = {
            'weight': weight,
            'ranges': sorted(ranges_for_indicator, key=lambda x: x[0]) # Ordenar por el valor mínimo
        }

# Función para obtener la puntuación de un indicador
def get_indicator_score(indicator_value, indicator_rules):
    if pd.isna(indicator_value):
        return 0 # Si el valor es NaN, se asigna 0 puntos (o se podría manejar de otra forma)

    for min_val, max_val, score in indicator_rules['ranges']:
        if min_val <= indicator_value <= max_val:
            return score
    return 0 # Si no coincide con ningún rango, asignar 0

print("Reglas de evaluación cargadas y parseadas:")
for indicator, rules in evaluation_rules.items():
    print(f"- {indicator}: Peso = {rules['weight']}%, Rangos = {rules['ranges']}")

Reglas de evaluación cargadas y parseadas:
- Net Profit/Mes: Peso = 30%, Rangos = [(-inf, 100.0, 0), (200.0, 500.0, 1), (500.0, 1000.0, 2), (1000.0, inf, 3)]
- PF: Peso = 25%, Rangos = [(-inf, 1.2, 0), (1.2, 1.49, 1), (1.5, 1.99, 2), (2.0, inf, 3)]
- WR: Peso = 15%, Rangos = [(-inf, 35.0, 0), (35.0, 49.0, 1), (50.0, 64.0, 2), (65.0, inf, 3)]
- Recovery Factor: Peso = 15%, Rangos = [(-inf, 1.0, 0), (1.0, 1.99, 1), (2.0, 3.99, 2), (4.0, inf, 3)]
- PayoffRatio: Peso = 15%, Rangos = [(-inf, 1.2, 0), (1.2, 1.79, 1), (1.8, 2.99, 2), (3.0, inf, 3)]


In [ ]:
# Celda 14 - Aplicar la lógica de scoring y calcular la Calificación Final

# Asegurarse de que `updated_master_df` sea el DataFrame sobre el que vamos a trabajar
# Si no está definida, se asume que final_df_to_add es la base si es que las celdas previas no se corrieron
if 'updated_master_df' not in locals() and 'updated_master_df' not in globals():
    print("Advertencia: 'updated_master_df' no encontrada. Utilizando 'final_df_to_add' como base.")
    bots_df = final_df_to_add.copy()
else:
    bots_df = updated_master_df.copy()

# Crear columnas para las puntuaciones individuales de cada indicador
for indicator_col_name in evaluation_rules.keys():
    score_col_name = f'Score {indicator_col_name}'
    bots_df[score_col_name] = bots_df[indicator_col_name].apply(lambda x: get_indicator_score(x, evaluation_rules[indicator_col_name]))

# Calcular la Calificación Final Ponderada
bots_df['Calificación Final'] = 0.0
for indicator_col_name, rules in evaluation_rules.items():
    score_col_name = f'Score {indicator_col_name}'
    if score_col_name in bots_df.columns:
        # Normalizar el peso para que la suma total de pesos sea 100
        normalized_weight = rules['weight'] / 100.0
        bots_df['Calificación Final'] += bots_df[score_col_name] * normalized_weight

print("Tabla Maestra con puntuaciones de indicadores y Calificación Final:")
display(bots_df[['Robot', 'Numero del Test'] + [f'Score {ind}' for ind in evaluation_rules.keys()] + ['Calificación Final']].head())

# Opcional: Guardar el DataFrame actualizado con la calificación final
# Asegurarse de que el path a la tabla maestra sea correcto
if 'MASTER_TABLE_PATH' in locals() or 'MASTER_TABLE_PATH' in globals():
    # Asegurar que las columnas existan antes de guardar
    all_cols_to_save = list(updated_master_df.columns) # Start with original columns
    for indicator_col_name in evaluation_rules.keys():
        score_col_name = f'Score {indicator_col_name}'
        if score_col_name not in all_cols_to_save:
            all_cols_to_save.append(score_col_name)
    if 'Calificación Final' not in all_cols_to_save:
        all_cols_to_save.append('Calificación Final')

    # Reordenar las columnas para poner las de puntuación y calificación al final
    final_cols = [col for col in bots_df.columns if col not in ['Calificación Final'] and not col.startswith('Score ')]
    final_cols.extend([f'Score {ind}' for ind in evaluation_rules.keys()])
    final_cols.append('Calificación Final')

    bots_df[final_cols].to_csv(MASTER_TABLE_PATH, index=False)
    print(f"Tabla maestra actualizada con las calificaciones y guardada en: {MASTER_TABLE_PATH}")
else:
    print("Advertencia: MASTER_TABLE_PATH no definida. No se pudo guardar la tabla maestra actualizada.")

Tabla Maestra con puntuaciones de indicadores y Calificación Final:


,Robot,Numero del Test,Score Net Profit/Mes,Score PF,Score WR,Score Recovery Factor,Score PayoffRatio,Calificación Final
0,RB001,T001,2,0,1,0,3,1.2


Tabla maestra actualizada con las calificaciones y guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv


## Análisis Detallado de la Calificación de Bots

Esta sección proporciona una visualización detallada de cómo cada indicador de los bots fue calificado, mostrando el valor del bot, el rango de evaluación aplicable y la puntuación obtenida, así como la contribución de cada indicador a la Calificación Final.

In [ ]:
# Celda 15 - Mostrar el análisis detallado de la calificación de cada bot

print("\n--- Análisis Detallado de la Calificación de Bots ---")

for idx, bot_row in bots_df.iterrows():
    robot_name = bot_row['Robot']
    test_number = bot_row['Numero del Test']
    final_score = bot_row['Calificación Final']

    print(f"\nRobot: {robot_name}, Test: {test_number}")
    print(f"Calificación Final: {final_score:.2f}\n")

    print("  Indicador             Valor Bot       Rango de Evaluación      Score   Peso (%)   Contribución")
    print("  --------------------  --------------  -----------------------  -----  ---------  ------------")

    for indicator_col_name, rules in evaluation_rules.items():
        bot_value = bot_row[indicator_col_name]
        indicator_weight = rules['weight']
        assigned_score = bot_row[f'Score {indicator_col_name}']

        # Encontrar el rango correspondiente al score asignado
        applicable_range_str = "N/A"
        for min_val, max_val, score_in_rule in rules['ranges']:
            if assigned_score == score_in_rule:
                # Reconstruir el string del rango original para mejor visualización
                # Esto es una simplificación, ya que el parse_range_string original pierde el formato exacto
                # del string, pero podemos aproximarlo o usar una lógica más compleja si es necesario.
                if min_val == -float('inf'):
                    applicable_range_str = f"< {max_val}"
                elif max_val == float('inf'):
                    applicable_range_str = f">= {min_val}"
                else:
                    applicable_range_str = f"{min_val} - {max_val}"
                # Si el valor real del bot es NaN, el score es 0, y el rango puede ser el de score 0
                if pd.isna(bot_value) and assigned_score == 0:
                    applicable_range_str = f"Valor NaN / Score 0"
                elif not pd.isna(bot_value):
                     # Para ser más precisos, si el valor del bot está en este rango
                     if min_val <= bot_value <= max_val:
                         break # Found the exact range the bot's value fell into
                     else:
                         # If we're on score 0 but the value is not in its range
                         # and it's not NaN, it means it didn't fit other ranges either.
                         # This is a fallback to ensure we show something sensible if initial matching was imperfect.
                         if assigned_score == 0 and not pd.isna(bot_value):
                             applicable_range_str = f"No coincide con rango definido (Score 0)"

        weighted_contribution = (assigned_score * (indicator_weight / 100.0))

        print(f"  {indicator_col_name:<20}  {str(bot_value):<14.4}  {applicable_range_str:<23}  {assigned_score:<5}  {indicator_weight:<9}  {weighted_contribution:<12.2f}")
    print("  ------------------------------------------------------------------------------------------------")


--- Análisis Detallado de la Calificación de Bots ---

Robot: RB001, Test: T001
Calificación Final: 1.20

  Indicador             Valor Bot       Rango de Evaluación      Score   Peso (%)   Contribución
  --------------------  --------------  -----------------------  -----  ---------  ------------
  Net Profit/Mes        720.            500.0 - 1000.0           2      30         0.60        
  PF                    1.1             < 1.2                    0      25         0.00        
  WR                    42.1            35.0 - 49.0              1      15         0.15        
  Recovery Factor       0.21            < 1.0                    0      15         0.00        
  PayoffRatio           4.41            >= 3.0                   3      15         0.45        
  ------------------------------------------------------------------------------------------------


## Guardar la Tabla Maestra con Calificaciones (Tabla Maestra 2)

Guardamos el DataFrame actualizado, que incluye las puntuaciones individuales de los indicadores y la `Calificación Final` para cada bot, en un nuevo archivo CSV.

In [ ]:
# Celda 16 - Guardar la Tabla Maestra con Calificaciones en un nuevo archivo

NEW_MASTER_TABLE_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra 2.csv'

# Asegurarse de que `bots_df` sea el DataFrame correcto a guardar
# bots_df ya debería estar actualizado con las columnas de Score y Calificación Final de la Celda 14

# Reordenar las columnas para poner las de puntuación y calificación al final si aún no están
current_cols = bots_df.columns.tolist()
score_cols = [col for col in current_cols if col.startswith('Score ')]
other_cols = [col for col in current_cols if not col.startswith('Score ') and col != 'Calificación Final']

final_save_cols_order = other_cols + score_cols + ['Calificación Final']

# Asegurarse de que todas las columnas que queremos estén presentes antes de reordenar y guardar
for col in final_save_cols_order:
    if col not in bots_df.columns:
        bots_df[col] = pd.NA # Añadir columnas faltantes si es el caso (aunque no debería ocurrir aquí)

# Guardar el DataFrame
bots_df[final_save_cols_order].to_csv(NEW_MASTER_TABLE_PATH, index=False)

print(f"Tabla maestra con calificaciones guardada exitosamente en: {NEW_MASTER_TABLE_PATH}")
display(bots_df[final_save_cols_order].head())

Tabla maestra con calificaciones guardada exitosamente en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra 2.csv


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Score Net Profit/Mes,Score PF,Score WR,Score Recovery Factor,Score PayoffRatio,Calificación Final
0,RB001,T001,2026-05-25 00:00:00,2026-06-03 00:00:00,MNQ,0.295664,50.0,213.0,720.413333,1.1,42.14,-995.6,0.213941,4.419704,224.3,-50.75,2,0,1,0,3,1.2


---